Data source: https://grouplens.org/datasets/hetrec-2011/

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px

from IPython.display import display
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

pd.options.display.max_columns = None
pd.options.display.precision = 2


## Interactions / User / Item

In [5]:
interaction_df = pd.read_table("user_ratedmovies.dat")
interaction_df.head()
interaction_df.dtypes
interaction_df.describe()

,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second
0,75,3,1.0,29,10,2006,23,17,16
1,75,32,4.5,29,10,2006,23,23,44
2,75,110,4.0,29,10,2006,23,30,8
3,75,160,2.0,29,10,2006,23,16,52
4,75,163,4.0,29,10,2006,23,29,30


userID           int64
movieID          int64
rating         float64
date_day         int64
date_month       int64
date_year        int64
date_hour        int64
date_minute      int64
date_second      int64
dtype: object

,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second
count,855598.00,855598.00,855598.00,855598.00,855598.00,855598.00,855598.00,855598.00,855598.00
mean,35190.83,8710.18,3.44,15.57,6.54,2005.32,12.12,29.65,29.51
std,20385.00,14446.85,1.00,8.95,3.51,2.28,7.60,17.27,17.31
min,75.00,1.00,0.50,1.00,1.00,1997.00,0.00,0.00,0.00
25%,18161.00,1367.00,3.00,8.00,4.00,2004.00,5.00,15.00,15.00
50%,33866.00,3249.00,3.50,15.00,7.00,2006.00,13.00,30.00,30.00
75%,52004.00,6534.00,4.00,23.00,10.00,2007.00,19.00,45.00,44.00
max,71534.00,65133.00,5.00,31.00,12.00,2009.00,23.00,59.00,59.00


In [6]:
interaction_df[["userID", "movieID"]].nunique()

userID      2113
movieID    10109
dtype: int64

### EDA On `Rating`

Insights:
- `2005-2008` have sufficient data for experiments
- `rating=4.0` is the best threshold to define positive/negative samples (for now)

In [7]:
# interaction_df[["userID", "movieID", "rating"]].nunique()
print("""
1997 - 2002: no "0.5" scale
2003, 2004, 2009: insufficient data
2005-2008: more data for experiments
------------------------
"""
)

print("Rating/interactions count for each year")
(
    interaction_df.groupby(
        ["date_year", "rating"]
    ).agg("count")
    .iloc[:, 0]
    .reset_index([-1, -2])
    .pivot_table(values="userID", index="rating", columns="date_year")
    .fillna(0)
    .astype("int")
)

print("---"*10)
print("The total ratings count in each year.")
pd.DataFrame(interaction_df["date_year"].value_counts().sort_index()).T

print("---"*10)
print("Rating count by month, 2005-2008")
(
    interaction_df[interaction_df["date_year"].between(2005, 2008)]
    .assign(
        date_year=interaction_df["date_year"].astype("str"),
        date_month=interaction_df["date_month"].astype("str").str.zfill(2),
    )
    .groupby(["date_year", "date_month"])
    .agg(rating_cnt=("rating", lambda x: x.count()))
    .reset_index([-1, -2])
    .pivot(columns="date_month", index="date_year")
)


1997 - 2002: no "0.5" scale
2003, 2004, 2009: insufficient data
2005-2008: more data for experiments
------------------------

Rating/interactions count for each year


date_year,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009
rating,,,,,,,,,,,,,
0.5,0,0,0,0,0,0,817,1690,2232,3486,3067,2148,48
1.0,24,99,643,1229,1300,1608,1447,1765,2794,4390,3457,2704,75
1.5,0,0,0,0,0,0,1272,2090,3307,4612,3989,2986,72
2.0,74,222,1627,3503,3493,4183,4059,4660,7649,10975,9279,7290,174
2.5,0,0,0,0,0,0,4274,7666,11386,14913,13246,10699,270
3.0,202,632,3585,8778,8034,10412,9694,13700,21693,30486,25411,22693,598
3.5,0,0,0,0,0,0,9063,17886,26809,36506,30262,29408,648
4.0,274,822,4473,12363,9714,12236,13189,19129,29499,42418,36351,34531,774
4.5,0,0,0,0,0,0,5976,9790,14178,21792,18839,17764,313


------------------------------
The total ratings count in each year.


date_year,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009
count,697,2110,12668,31860,26732,32813,54665,82929,127411,183785,156422,140401,3105


------------------------------
Rating count by month, 2005-2008


rating_cnt                                                          \
date_month         01     02     03     04     05     06     07     08     09   
date_year                                                                       
2005             9830   8274  12253  14082  12532  12714   9258   7692   8224   
2006            19975  15531  13177  17644  13229  13806  20644  14818  12982   
2007            14387  14664  14251  13940  10124  13392  13255  15685   9545   
2008            15016  10980  11540   8739   9231   9255  12922  15385   8736   

                                 
date_month     10     11     12  
date_year                        
2005         9910  11408  11234  
2006        12472  11250  18257  
2007        11218  11058  14903  
2008        11596  11962  15039

In [8]:
px.bar(
    (
        interaction_df.assign(date_year=interaction_df["date_year"].astype("str"))
        [interaction_df["date_year"].between(2005, 2008)]
        .groupby(
            ["date_year", "rating"]
        ).count()
        .reset_index([-1, -2])
        .rename(columns={"userID": "cnt"})
        [["date_year", "rating", "cnt"]]
    ),
    x="rating", y="cnt", 
    color="date_year", 
    color_discrete_sequence=px.colors.qualitative.Bold,
    barmode="group",
    height=400,
    title="Ratings cnt (2005 - 2008)"
)

In [9]:
px.bar(
    (
        interaction_df.assign(
            date_year=interaction_df["date_year"].astype("str"),
            date_month=interaction_df["date_month"].astype("str").str.zfill(2),
        )
        [interaction_df["date_year"].between(2005, 2008)]
        .groupby(
            ["date_year", "date_month", "rating"]
        ).count()
        .reset_index([-1, -2, -3])
        .rename(columns={"userID": "cnt"})
        [["date_year", "date_month", "rating", "cnt"]]
        .sort_values(["date_year", "date_month"])
    ),
    x="rating", y="cnt", 
    facet_col="date_month", facet_row="date_year", 
    barmode="group",
    height=800, width=2000,
    title="Ratings dist. (detailed) (2005 - 2008)"
)

#### `Rating`: threshold of positive/negative samples

In [10]:
# TODO: change the threshold to check out effects of the cutting point
THRESHOLD = 4.0

px.pie(
    (
        interaction_df[interaction_df["date_year"].between(2005, 2008)]
        .assign(
            date_year=interaction_df["date_year"].astype("str"),
            action_type=interaction_df["rating"].apply(lambda x: "positive" if x >= THRESHOLD else "negative")
        )
        .groupby(
            ["date_year", "action_type"]
        ).count()
        .reset_index([-1, -2])
        .rename(columns={"userID": "cnt"})
        [["date_year", "action_type", "cnt"]]
    ),
    names="action_type",
    values="cnt",
    color_discrete_sequence=px.colors.qualitative.Prism,
    facet_col="date_year",
    height=375,
    title=f"Action type with threshold={THRESHOLD} (2005 - 2008)",
)

In [11]:
# TODO: change the threshold to check out effects of the cutting point
THRESHOLD = 4.0

px.pie(
    (
        interaction_df[interaction_df["date_year"].between(2005, 2008)]
        .assign(
            date_year=interaction_df["date_year"].astype("str"),
            date_month=interaction_df["date_month"].astype("str").str.zfill(2),
            action_type=interaction_df["rating"].apply(lambda x: "positive" if x >= THRESHOLD else "negative")
        )
        .groupby(
            ["date_year", "date_month", "action_type"]
        ).agg(cnt=("userID", lambda x: x.count()))
        .reset_index([-1, -2, -3])
        [["date_year", "date_month", "action_type", "cnt"]]
    ),
    names="action_type",
    values="cnt",
    color_discrete_sequence=px.colors.qualitative.Prism,
    facet_col="date_month", facet_row="date_year",
    height=700, width=2000,
    title=f"Action type with threshold={THRESHOLD} by month (2005 - 2008)",
)

In [12]:
(
    interaction_df[interaction_df["date_year"].between(2006, 2008)]
    .groupby(["date_year", "date_month"])
    .count()
    .iloc[:, :1]
    .T
)

date_year    2006                                                          \
date_month     1      2      3      4      5      6      7      8      9    
userID      19975  15531  13177  17644  13229  13806  20644  14818  12982   

date_year                         2007                                     \
date_month     10     11     12     1      2      3      4      5      6    
userID      12472  11250  18257  14387  14664  14251  13940  10124  13392   

date_year                                             2008                \
date_month     7      8     9      10     11     12     1      2      3    
userID      13255  15685  9545  11218  11058  14903  15016  10980  11540   

date_year                                                              
date_month    4     5     6      7      8     9      10     11     12  
userID      8739  9231  9255  12922  15385  8736  11596  11962  15039

In [13]:
# 2006 - 2008: 480,608 total
# 2006 - 2007: 340,207 (70.8%) train
# 2008 01 - 06: 64,761 (13.5%) valid
# 2008 07 - 12: 75,640 (15.7%) test
len(
    interaction_df[
        (interaction_df["date_year"] == 2008) &
        (interaction_df["date_month"].between(7, 12))
    ]
)



75640

### EDA on `User` & `Item`
Insights: `[2005-2008]`

In [14]:
interaction_df[["userID", "movieID"]].nunique()

userID      2113
movieID    10109
dtype: int64

#### User
- 一年大約超過 1k unique users -> 一個月約 400~500
- `2005` 的人數分佈和 `2006-2008` 不太一致
- 大約 10~20% light users (平均 < 20 次/年)
- 整體評分以 3.5 為中心形成一個鐘型的常態分佈（由上面分析，從數量觀點來看以 `4.0` 切分）

In [15]:
print("Unique user count for each year") 
tmp_unique_user_df = pd.DataFrame(interaction_df.groupby(["date_year"]).agg(uu_cnt=("userID", lambda x: x.nunique()))).T
tmp_unique_user_df
px.line(
    tmp_unique_user_df.T.reset_index(-1),
    x="date_year", y="uu_cnt",
    height=300, width=1000,
    title="Unique user count (1997-2009)",
)
print("---"*10)

print("Unique user count by month, 2005-2008")
tmp_unique_user_month_df = (
    interaction_df[interaction_df["date_year"].between(2005, 2008)]
    .assign(
        date_year=interaction_df["date_year"].astype("str"),
        date_month=interaction_df["date_month"].astype("str").str.zfill(2),
    )
    .groupby(["date_year", "date_month"])
    .agg(uu_cnt=("userID", lambda x: x.nunique()))
    .reset_index([-1, -2])
)
tmp_unique_user_month_df.pivot(columns="date_month", index="date_year")
px.line(
    tmp_unique_user_month_df,
    x="date_month", y="uu_cnt", 
    color="date_year", color_discrete_sequence=px.colors.qualitative.Bold,
    height=300, width=1000,
    title="Unique user count by month (2005-2008)",
)

Unique user count for each year


date_year,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009
uu_cnt,12,23,79,163,209,246,325,435,668,1259,1244,1194,209


------------------------------
Unique user count by month, 2005-2008


uu_cnt                                                       
date_month     01   02   03   04   05   06   07   08   09   10   11   12
date_year                                                               
2005          296  276  301  310  339  358  384  353  359  370  389  412
2006          520  516  486  478  443  445  488  491  469  489  489  510
2007          522  509  518  496  477  456  467  499  450  439  432  454
2008          509  463  509  445  448  414  446  491  422  428  470  469

In [16]:
# avg. rating count for each users
def calculate_percentile(group, percentile):
    return np.round(np.percentile(group, percentile), 2)

tmp_rating_by_year_user_df = (
    interaction_df[interaction_df["date_year"].between(2005, 2008)]
    .groupby(["date_year", "userID"])
    .agg(
        rating_cnt=("rating", lambda x: x.count()),  # by user
        rating_score=("rating", lambda x: x.mean()), # by user
    )
    .reset_index([-1, -2])
)
tmp_rating_by_year_user_df
print("---"*10)

# NOTE: 這邊看的 rating_cnt 是指「各年份」「平均每個使用者」的「給分次數」
# NOTE: 觀察點：使用者普遍使用頻率的習慣偏好
COL_NAME = "rating_cnt"
print(f"{COL_NAME} statistics on users")
(
    tmp_rating_by_year_user_df[["date_year", "userID", COL_NAME]].groupby("date_year")
    .agg(
        count=("userID", "count"),
        mean=(COL_NAME, "mean"),
        min=(COL_NAME, "min"),
        percentile_25=(COL_NAME, lambda x: calculate_percentile(x, 25)),
        percentile_50=(COL_NAME, lambda x: calculate_percentile(x, 50)),   # Median
        percentile_75=(COL_NAME, lambda x: calculate_percentile(x, 75)),
        percentile_90=(COL_NAME, lambda x: calculate_percentile(x, 90)),
        percentile_95=(COL_NAME, lambda x: calculate_percentile(x, 95)),
        percentile_99=(COL_NAME, lambda x: calculate_percentile(x, 99)),
        max=(COL_NAME, "max"),
    ).T
)

px.histogram(
    tmp_rating_by_year_user_df, x="rating_cnt",
    facet_row="date_year",
    height=600, width=1000,
    title="Distribution of Avg. Ratings Count per User "
)
print("---"*10)

# NOTE: 這邊看的 rating_score 是指「各年份」「平均每個使用者」會給出的「平均分數」
# NOTE: 觀察點：使用者普遍給分的習慣偏好（先針對使用者自己的給分做平均，有幫助減少 heavy user 對此議題產生的數量偏誤）
# TODO: rating_score 可能不應該 by user 算，可以直接看當年份平均分數都給多少，
COL_NAME = "rating_score"
print(f"{COL_NAME} statistics on users")
(
    tmp_rating_by_year_user_df[["date_year", COL_NAME]].groupby("date_year")
    .agg(
        mean=(COL_NAME, "mean"),
        min=(COL_NAME, "min"),
        percentile_25=(COL_NAME, lambda x: calculate_percentile(x, 25)),
        percentile_50=(COL_NAME, lambda x: calculate_percentile(x, 50)),   # Median
        percentile_75=(COL_NAME, lambda x: calculate_percentile(x, 75)),
        percentile_90=(COL_NAME, lambda x: calculate_percentile(x, 90)),
        max=(COL_NAME, "max"),
    ).T
)

px.histogram(
    tmp_rating_by_year_user_df, x="rating_score",
    facet_row="date_year",
    height=600, width=1000,
    title="Distribution of Avg. Ratings Score per User"
)

px.histogram(
    interaction_df[interaction_df["date_year"].between(2005, 2008)],
    x="rating",
    facet_row="date_year",
    height=600, width=1000,
    title="Distribution of Overall Ratings Score"
)

,date_year,userID,rating_cnt,rating_score
0,2005,175,226,4.27
1,2005,190,34,3.24
2,2005,267,41,2.89
3,2005,325,209,3.63
4,2005,476,29,3.91
...,...,...,...,...
4360,2008,71478,7,3.36
4361,2008,71487,176,3.82
4362,2008,71497,95,3.48
4363,2008,71509,210,3.80


------------------------------
rating_cnt statistics on users


date_year,2005,2006,2007,2008
count,668.00,1259.00,1244.00,1194.00
mean,190.74,145.98,125.74,117.59
min,1.00,1.00,1.00,1.00
percentile_25,49.00,35.00,25.75,23.00
percentile_50,108.50,75.00,59.50,58.00
percentile_75,228.00,170.00,139.00,138.75
percentile_90,468.50,356.00,316.70,293.70
percentile_95,649.30,542.50,502.00,454.70
percentile_99,1141.85,955.36,955.23,768.56
max,2250.00,2768.00,1600.00,1562.00


------------------------------
rating_score statistics on users


date_year,2005,2006,2007,2008
mean,3.53,3.55,3.54,3.59
min,0.50,0.50,0.50,1.70
percentile_25,3.24,3.26,3.25,3.32
percentile_50,3.55,3.57,3.57,3.58
percentile_75,3.80,3.85,3.84,3.88
percentile_90,4.04,4.11,4.11,4.17
max,5.00,5.00,5.00,5.00


#### Item
- 一年大約 7k unique items -> 一個月大約 ~3k unique items
- 在這四年區間中沒有明顯的 patterns (可能跟每年上映電影的數量有關)


In [17]:
print("Unique item count for each year") 
tmp_unique_item_df = pd.DataFrame(interaction_df.groupby(["date_year"]).agg(movie_cnt=("movieID", lambda x: x.nunique()))).T
tmp_unique_item_df
px.line(
    tmp_unique_item_df.T.reset_index(-1),
    x="date_year", y="movie_cnt",
    height=300, width=1000,
    title="Unique item count (1997-2009)",
)
print("---"*10)


print("Unique movie count by month, 2005-2008")
tmp_unique_item_month_df = (
    interaction_df[interaction_df["date_year"].between(2005, 2008)]
    .assign(
        date_year=interaction_df["date_year"].astype("str"),
        date_month=interaction_df["date_month"].astype("str").str.zfill(2),
    )
    .groupby(["date_year", "date_month"])
    .agg(movie_cnt=("movieID", lambda x: x.nunique()))
    .reset_index([-1, -2])
)
tmp_unique_item_month_df.pivot(columns="date_month", index="date_year")
px.line(
    tmp_unique_item_month_df,
    x="date_month", y="movie_cnt", 
    color="date_year", color_discrete_sequence=px.colors.qualitative.Bold,
    height=300, width=1000,
    title="Unique item count by month (2005-2008)",
)

Unique item count for each year


date_year,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009
movie_cnt,433,861,2195,3084,3573,4274,5495,6177,6825,7414,7679,7812,1708


------------------------------
Unique movie count by month, 2005-2008


movie_cnt                                                        \
date_month        01    02    03    04    05    06    07    08    09    10   
date_year                                                                    
2005            3211  3520  3622  3298  2999  3483  3123  2886  3035  2888   
2006            3916  3672  3519  3968  3272  3987  3867  3825  3153  3230   
2007            3314  3856  4002  3365  3195  3771  3419  3738  2644  3237   
2008            3626  3140  3128  2749  2929  3039  3476  3573  2981  3082   

                        
date_month    11    12  
date_year               
2005        3232  3266  
2006        3054  4158  
2007        3394  3907  
2008        3408  4343

In [18]:
# Cold start items study
tmp_rating_by_year_item_df = (
    interaction_df[interaction_df["date_year"].between(2005, 2008)]
    .groupby(["date_year", "movieID"])
    .agg(
        rating_cnt=("rating", lambda x: x.count()),  # by item
        rating_score=("rating", lambda x: x.mean()), # by item
    )
    .reset_index([-1, -2])
)
tmp_rating_by_year_item_df
print("---"*10)

px.histogram(
    tmp_rating_by_year_user_df, x="rating_cnt",
    facet_row="date_year",
    height=600, width=1000,
    title="Distribution of Avg. Ratings Count per Item"
)



,date_year,movieID,rating_cnt,rating_score
0,2005,1,169,3.67
1,2005,2,129,2.97
2,2005,3,47,2.69
3,2005,4,2,1.50
4,2005,5,37,2.66
...,...,...,...,...
29725,2008,64993,1,3.00
29726,2008,64997,3,3.33
29727,2008,64999,1,0.50
29728,2008,65006,1,4.00


------------------------------


## Item Attributes
- actors
- countries
- directors
- genres
- locations
- tags

#### Actors
- 總共有 10,174 部電影的演員資料，每部電影紀錄超過 10 位演員資料
- `ranking` 欄位指出在電影中的貢獻程度，可以只取前三名來製作 diversity preference
- ~20 部電影沒有演員資料（2006-2008 interactions）

In [258]:
actor_df = pd.read_table("movie_actors.dat", encoding='latin-1')
actor_df.head()
actor_df.nunique()

# 每部電影的演員名單（照貢獻程度排序）
actor_df.sort_values(["movieID", "ranking"]).head(10)


,movieID,actorID,actorName,ranking
0,1,annie_potts,Annie Potts,10
1,1,bill_farmer,Bill Farmer,20
2,1,don_rickles,Don Rickles,3
3,1,erik_von_detten,Erik von Detten,13
4,1,greg-berg,Greg Berg,17


movieID      10174
actorID      95321
actorName    95241
ranking        220
dtype: int64

,movieID,actorID,actorName,ranking
22,1,tom_hanks,Tom Hanks,1
21,1,tim_allen,Tim Allen,2
2,1,don_rickles,Don Rickles,3
7,1,jim_varney,Jim Varney,4
23,1,wallace_shawn,Wallace Shawn,5
5,1,jack_angel,Jack Angel,6
20,1,sherry_lynn,Sherry Lynn,7
13,1,laurie_metcalf,Laurie Metcalf,8
14,1,patrick_pinney,Patrick Pinney,9
0,1,annie_potts,Annie Potts,10


In [266]:
interaction_movie_set = set(interaction_df[interaction_df["date_year"].between(2006, 2008)]["movieID"].values)
actor_movie_set = set(actor_df["movieID"].values)
print("Movies without actors data:", len(interaction_movie_set - actor_movie_set))

Movies without actors data: 20


### Countries
- 總共有 71 個國家，其中 USA 佔最大宗
- 全部的電影都有國家資訊

In [22]:
country_df = pd.read_table("movie_countries.dat")
country_df.head()
country_df.nunique()
country_df["country"].value_counts().head()

,movieID,country
0,1,USA
1,2,USA
2,3,USA
3,4,USA
4,5,USA


movieID    10197
country       71
dtype: int64

country
USA       6831
UK        1015
France     577
Canada     233
Italy      205
Name: count, dtype: int64

In [23]:
interaction_movie_set = set(interaction_df[interaction_df["date_year"].between(2006, 2008)]["movieID"].values)
country_movie_set = set(country_df["movieID"].values)
print("Movies without country data:", len(interaction_movie_set - country_movie_set))

Movies without country data: 0


#### directors
- 總共有 10,155 部電影的導演資料，每部電影只有一個導演
- ~40 部電影沒有導演資訊

In [ ]:
director_df = pd.read_table("movie_directors.dat", encoding='latin-1')
director_df.head()
print("data count:", len(director_df))
director_df.nunique()

,movieID,directorID,directorName
0,1,john_lasseter,John Lasseter
1,2,joe_johnston,Joe Johnston
2,3,donald_petrie,Donald Petrie
3,4,forest_whitaker,Forest Whitaker
4,5,charles_shyer,Charles Shyer


data count: 10155


movieID         10155
directorID       4060
directorName     4053
dtype: int64

In [270]:
interaction_movie_set = set(interaction_df[interaction_df["date_year"].between(2006, 2008)]["movieID"].values)
director_movie_set = set(director_df["movieID"].values)
print("Movies without director data:", len(interaction_movie_set - director_movie_set))

Movies without director data: 40


In [ ]:
# NOTE: actor, director 分別缺失的電影幾乎不重疊
set_a = (interaction_movie_set - actor_movie_set)
set_b = (interaction_movie_set - director_movie_set)
print(len(set_a), len(set_b))
len(set_a.union(set_b))


20 40


58

#### genres
- 總共有 10,197 部電影的種類資訊，每部電影的種類數不一，共有 20 個類別
- 全部電影都有種類資料
- 一部電影 max: 8, min: 1, mean/median: 2

In [283]:
genres_df = pd.read_table("movie_genres.dat")
genres_df.head()
genres_df.nunique()
print(genres_df["genre"].unique())

,movieID,genre
0,1,Adventure
1,1,Animation
2,1,Children
3,1,Comedy
4,1,Fantasy


movieID    10197
genre         20
dtype: int64

['Adventure' 'Animation' 'Children' 'Comedy' 'Fantasy' 'Romance' 'Drama'
 'Action' 'Crime' 'Thriller' 'Horror' 'Mystery' 'Sci-Fi' 'IMAX'
 'Documentary' 'War' 'Musical' 'Film-Noir' 'Western' 'Short']


In [284]:
interaction_movie_set = set(interaction_df[interaction_df["date_year"].between(2006, 2008)]["movieID"].values)
genres_movie_set = set(genres_df["movieID"].values)
print("Movies without genres data:", len(interaction_movie_set - genres_movie_set))

Movies without genres data: 0


In [298]:
tmp_genres_by_movie_df = (
    genres_df
    .groupby(["movieID"])
    .agg(genre_cnt=("genre", lambda x: x.count()))  # by movie
    .reset_index([-1])
)
tmp_genres_by_movie_df.head()
print("---"*10)
tmp_genres_by_movie_df.describe()

px.histogram(
    tmp_genres_by_movie_df, x="genre_cnt",
    height=400, width=800,
    nbins=10,
    title="Distribution of Avg. Genre Count per Movie "
)
print("---"*10)

,movieID,genre_cnt
0,1,5
1,2,3
2,3,2
3,4,3
4,5,1


------------------------------


,movieID,genre_cnt
count,10197.00,10197.00
mean,12852.74,2.04
std,17431.00,1.03
min,1.00,1.00
25%,2780.00,1.00
50%,5421.00,2.00
75%,8664.00,3.00
max,65133.00,8.00


------------------------------


### locations （難使用）
- 一部電影有多筆 location 資料，也有 1,268 部電影沒有 location 資訊
- 全部的電影都有包含在這張表中

In [32]:
location_df = pd.read_table("movie_locations.dat", encoding='latin-1')
location_df.head()
location_df["movieID"].nunique()
print("movies with no locations info:", location_df["location1"].isna().sum())

,movieID,location1,location2,location3,location4
0,1,NaN,NaN,NaN,NaN
1,2,Canada,British Columbia,NaN,NaN
2,2,Canada,British Columbia,Delta,NaN
3,2,Canada,British Columbia,Delta,Tsawwassen
4,2,Canada,British Columbia,Maple Ridge,NaN


10197

movies with no locations info: 1268


In [34]:
interaction_movie_set = set(interaction_df[interaction_df["date_year"].between(2006, 2008)]["movieID"].values)
location_movie_set = set(location_df["movieID"].values)
print("Movies without location data:", len(interaction_movie_set - location_movie_set))

Movies without location data: 0


#### tags （不建議使用）
- 總共有 7,155 部電影的 tags （缺失很多: 2,579 部）
- tags 類似於關鍵字的標籤，以特定名詞來描述電影內容風格等，資訊量很 sparse
- 每部電影有多個 tags，其中 tags 有權重分別（不好處理），若真的要使用，應該要取每部電影最重要的 TopK 個 tags
- tagWeight 大多都是 1（沾邊），只有少數權重數字高（max: 42）

In [310]:
movie_tags_df = pd.read_table("movie_tags.dat")
movie_tags_df.head()
movie_tags_df.nunique()
movie_tags_df.sort_values(["movieID", "tagWeight"], ascending=[True, False]).head(20)
movie_tags_df.describe()

,movieID,tagID,tagWeight
0,1,7,1
1,1,13,3
2,1,25,3
3,1,55,3
4,1,60,1


movieID      7155
tagID        5297
tagWeight      30
dtype: int64

,movieID,tagID,tagWeight
11,1,465,17
6,1,326,10
19,1,900,9
23,1,2119,5
24,1,2293,4
1,1,13,3
2,1,25,3
3,1,55,3
9,1,382,2
10,1,459,2


,movieID,tagID,tagWeight
count,51795.00,51795.00,51795.00
mean,12163.18,4354.54,1.38
std,17136.00,4157.63,1.28
min,1.00,1.00,1.00
25%,1861.00,775.00,1.00
50%,4399.00,2738.00,1.00
75%,8494.00,6800.00,1.00
max,65130.00,16518.00,42.00


In [311]:
interaction_movie_set = set(interaction_df[interaction_df["date_year"].between(2006, 2008)]["movieID"].values)
tags_movie_set = set(movie_tags_df["movieID"].values)
print("Movies without tags data:", len(interaction_movie_set - tags_movie_set))

Movies without tags data: 2579


In [312]:
tags_df = pd.read_table("tags.dat", encoding='latin-1')
tags_df.head(10)
tags_df.nunique()

,id,value
0,1,earth
1,2,police
2,3,boxing
3,4,painter
4,5,whale
5,6,medieval
6,7,funny
7,8,almodovar
8,9,finnish
9,10,brothers quay


id       13222
value    13222
dtype: int64

#### others (movie.dat, user_taggedmovies.dat)
For `movie.dat`
- 似乎整合了爛番茄的電影資料（各種評分、留言數字）

For `user_taggedmovies.dat`
- 另一種使用者互動數據紀錄，在我們的使用情境中不考慮


In [ ]:
movie_df = pd.read_table("movies.dat", encoding="latin-1")
movie_df.head()

,id,title,imdbID,spanishTitle,imdbPictureURL,year,rtID,rtAllCriticsRating,rtAllCriticsNumReviews,rtAllCriticsNumFresh,rtAllCriticsNumRotten,rtAllCriticsScore,rtTopCriticsRating,rtTopCriticsNumReviews,rtTopCriticsNumFresh,rtTopCriticsNumRotten,rtTopCriticsScore,rtAudienceRating,rtAudienceNumRatings,rtAudienceScore,rtPictureURL
0,1,Toy story,114709,Toy story (juguetes),http://ia.media-imdb.com/images/M/MV5BMTMwNDU0...,1995,toy_story,9,73,73,0,100,8.5,17,17,0,100,3.7,102338,81,http://content7.flixster.com/movie/10/93/63/10...
1,2,Jumanji,113497,Jumanji,http://ia.media-imdb.com/images/M/MV5BMzM5NjE1...,1995,1068044-jumanji,5.6,28,13,15,46,5.8,5,2,3,40,3.2,44587,61,http://content8.flixster.com/movie/56/79/73/56...
2,3,Grumpy Old Men,107050,Dos viejos gruñones,http://ia.media-imdb.com/images/M/MV5BMTI5MTgy...,1993,grumpy_old_men,5.9,36,24,12,66,7,6,5,1,83,3.2,10489,66,http://content6.flixster.com/movie/25/60/25602...
3,4,Waiting to Exhale,114885,Esperando un respiro,http://ia.media-imdb.com/images/M/MV5BMTczMTMy...,1995,waiting_to_exhale,5.6,25,14,11,56,5.5,11,5,6,45,3.3,5666,79,http://content9.flixster.com/movie/10/94/17/10...
4,5,Father of the Bride Part II,113041,Vuelve el padre de la novia (Ahora también abu...,http://ia.media-imdb.com/images/M/MV5BMTg1NDc2...,1995,father_of_the_bride_part_ii,5.3,19,9,10,47,5.4,5,1,4,20,3,13761,64,http://content8.flixster.com/movie/25/54/25542...


In [316]:
user_tag_df = pd.read_table("user_taggedmovies.dat")
user_tag_df

,userID,movieID,tagID,date_day,date_month,date_year,date_hour,date_minute,date_second
0,75,353,5290,29,10,2006,23,20,15
1,78,4223,5264,16,4,2007,4,43,45
2,127,1343,1544,28,8,2007,3,42,27
3,127,1343,12330,28,8,2007,3,42,27
4,127,2080,1451,28,8,2007,3,42,47
...,...,...,...,...,...,...,...,...,...
47952,71534,7937,306,3,12,2007,3,7,14
47953,71534,8848,331,3,12,2007,3,6,19
47954,71534,8848,427,3,12,2007,3,6,27
47955,71534,25833,7671,3,12,2007,3,7,31
